# Fine-Tune Whisper for Moroccan Darija (TDD-01)

Trains `openai/whisper-small` on **DODa** (`atlasia/DODa-audio-dataset`) for the
Smart Airport Wayfinding Assistant — the project's owned ML contribution.

**Runs on a free GPU**, either way:
- **Browser Colab:** Runtime -> Change runtime type -> **T4 GPU**.
- **VS Code (Google Colab extension):** Select Kernel -> Colab -> Auto Connect (T4).

Just run the cells top to bottom. The heavy lifting is in the repo's `src/`
scripts — this notebook only orchestrates them on the GPU machine.

> The GPU machine is **temporary**: when the session ends it is wiped. Step 7
> pushes the trained model to the Hugging Face Hub so it survives.

## 1. Check the GPU
Confirms a GPU is attached. If this errors or shows no GPU, enable it first (see above).

In [ ]:
!nvidia-smi

## 2. Get the code
Clone the repo and move into the ASR package (all `src.` modules run from here).

In [ ]:
# If you already cloned in this session, this is a no-op.
import os
if not os.path.isdir('ML-21-Flughafen'):
    !git clone https://github.com/Oussamawork/ML-21-Flughafen.git
%cd ML-21-Flughafen/asr_finetuning
!pwd

## 3. Install dependencies
Installs transformers/datasets/etc. Colab already ships PyTorch + CUDA.

In [ ]:
!pip install -q -r requirements.txt

## 4. Log in to Hugging Face
DODa is **gated**. First, on huggingface.co, open the
[dataset page](https://huggingface.co/datasets/atlasia/DODa-audio-dataset) and
click **Agree / Access** while logged in. Then create a token at
`huggingface.co/settings/tokens` (role: *read*, or *write* if you will push the
model in step 7) and paste it below.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Smoke test (optional, ~a few min)
A tiny 5-step run to prove the whole pipeline works *before* the long job.
It downloads + caches DODa (reused by the real run) and trains on a handful of
samples. Success = it finishes without error and prints a WER number.

In [ ]:
!python -m src.train --config config/doda_darija.yaml \
    --dataset.max_train_samples 64 --dataset.max_eval_samples 16 \
    --training.max_steps 5 --training.eval_steps 5 --training.save_steps 5 \
    --training.warmup_steps 1 --training.logging_steps 1 \
    --training.output_dir ./outputs/smoke

## 6a. Baseline: evaluate the *un-tuned* model
Measure `whisper-small` BEFORE fine-tuning on the DODa eval split. Note this
WER/CER — the improvement over it is your headline result.

In [ ]:
!python -m src.evaluate_model --config config/doda_darija.yaml \
    --model.name openai/whisper-small

## 6b. Fine-tune (the long step, ~1-3 h on a T4)
Trains on DODa with the settings in `config/doda_darija.yaml`
(`max_steps=3000`, `batch=16`, fp16, best-checkpoint-by-WER). Eval WER prints
every 300 steps and should trend **down**. Leave the tab open.

**If you hit a CUDA out-of-memory error**, lower the batch size and compensate
with accumulation (uncomment the second line):

In [ ]:
!python -m src.train --config config/doda_darija.yaml

# OOM-safe variant (same effective batch size, less memory):
# !python -m src.train --config config/doda_darija.yaml \
#     --training.per_device_train_batch_size 8 --training.gradient_accumulation_steps 2

## 6c. Evaluate the fine-tuned model
Same eval split as the baseline -> compare the two WER/CER numbers. This delta
is the graded ML contribution. `evaluate_model` also prints a few REF/PRED
examples for the report.

In [ ]:
!python -m src.evaluate_model --config config/doda_darija.yaml \
    --model.name ./outputs/whisper-small-doda-darija

## 7. Save the model to the Hugging Face Hub
The trained model lives only on this temporary machine. Push it to your account
so it is kept and can be loaded by the backend later. Needs a **write** token
(step 4). Change the repo id to your username.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

CKPT = './outputs/whisper-small-doda-darija'
REPO = 'Oussamawork/whisper-small-darija'  # <-- change to your HF username/repo

WhisperForConditionalGeneration.from_pretrained(CKPT).push_to_hub(REPO)
WhisperProcessor.from_pretrained(CKPT).push_to_hub(REPO)
print('Pushed to https://huggingface.co/' + REPO)

## 8. Quick listen test (optional)
Transcribe one DODa clip with the fine-tuned model and compare to the reference
transcript — a sanity check that it actually learned Darija.

In [ ]:
from datasets import load_dataset, Audio
from src.transcribe import WhisperTranscriber

sample = load_dataset('atlasia/DODa-audio-dataset', split='train[:1]')\
    .cast_column('audio', Audio(sampling_rate=16000))[0]

tr = WhisperTranscriber('./outputs/whisper-small-doda-darija')
pred = tr.transcribe(sample['audio']['array'], sampling_rate=16000)
print('REF :', sample['darija_Arab_new'])
print('PRED:', pred)